# Stage 3 — Canonicalization

Map raw skill strings to a canonical vocabulary using exact + fuzzy matching.
Unmatched skills are reported for vocabulary expansion.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
sys.path.insert(0, str(Path(".").resolve()))

from src.utils import load_skill_vocabulary, canonicalize_skill_list

In [ ]:
df_jobs = pd.read_csv("data/interim/jobs_cleaned.csv")
df_placements = pd.read_csv("data/interim/placements_cleaned.csv")

import ast
df_jobs["skills_list"] = df_jobs["skills_list"].apply(ast.literal_eval)
df_placements["skills_list"] = df_placements["skills_list"].apply(ast.literal_eval)

## Load vocabulary

In [ ]:
vocab = load_skill_vocabulary("configs/skills_canonical.yaml")
print(f"Vocabulary size: {len(vocab)} aliases")
list(vocab.items())[:10]

## Canonicalize skills

In [ ]:
df_jobs["canonical_skills"] = df_jobs["skills_list"].apply(
    lambda s: canonicalize_skill_list(s, vocab, unmatched_policy="keep_raw")
)
df_placements["canonical_skills"] = df_placements["skills_list"].apply(
    lambda s: canonicalize_skill_list(s, vocab, unmatched_policy="keep_raw")
)
df_jobs[["job_id", "skills_list", "canonical_skills"]].head()

## Canonicalization report

In [ ]:
from collections import Counter
from src.utils.canonicalize import canonicalize_skill

all_raw = [s for lst in list(df_jobs["skills_list"]) + list(df_placements["skills_list"]) for s in lst]
counts = Counter(all_raw)
rows = []
for raw, cnt in counts.items():
    can = canonicalize_skill(raw, vocab)
    tier = "exact" if raw.lower() in vocab else ("fuzzy" if can else "unmatched")
    rows.append({"raw_skill": raw, "canonical_skill": can, "match_tier": tier, "count": cnt})
report = pd.DataFrame(rows)
print(report.groupby("match_tier")["raw_skill"].count())
report.to_csv("data/interim/canonicalization_report.csv", index=False)
report.head(15)

## Save canonicalized checkpoints

In [ ]:
df_jobs.to_csv("data/interim/jobs_canonicalized.csv", index=False)
df_placements.to_csv("data/interim/placements_canonicalized.csv", index=False)
print("Done.")